PySpark merupakan antarmuka Python dari Apache Spark yang memungkinkan pemrosesan data berskala besar secara paralel dan terdistribusi. Spark dirancang untuk menangani big data dengan performa tinggi melalui mekanisme in-memory computing, sehingga lebih cepat dibanding pendekatan tradisional seperti Hadoop MapReduce. File ini merupakan implementasi dari penggunaan Pyspark.

Pada project ini akan dilakukan klasifikasi Price_categori dengan menggunakan fitur Brand, Body_Type, Fuel_Type, Transmission, Manufacturing_Country, Age_Category, Engine_CC, Horsepower, Mileage_km_per_l, Car_Age, HP_per_CC,
Efficiency_Score. Adapun fitur yang tidak di gunakan yaitu cars_id dan Price_USD


In [ ]:
!pip install pyspark



Kode ini digunakan untuk mengimpor `SparkSession`, yaitu komponen utama dalam PySpark yang berfungsi untuk memulai dan mengelola aplikasi Apache Spark.

`SparkSession` digunakan sebagai penghubung antara program Python dengan Spark Engine dalam proses pengolahan data terdistribusi.


In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import when

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    RandomForestClassifier,

)

from pyspark.ml import Pipeline

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

Kode ini digunakan untuk menginisialisasi Spark Session sebagai lingkungan kerja Apache Spark. Nama aplikasi diberikan dengan "Car Price Classification" untuk identifikasi proses pengolahan data yang dijalankan.

In [ ]:
spark = SparkSession.builder \
    .appName("Car Price Classification") \
    .getOrCreate()

Kode ini digunakan untuk membaca file dataset CSV menggunakan Apache Spark dan menyimpannya ke dalam dataframe df.

Dataset yang digunakan adalah dataset "global_cars_enhanced.csv" yang berisikan multivariate dan digunakan untuk klasifikasi sebagai tujuan dari implementasi proyek ini.

In [ ]:
from google.colab import files
uploaded = files.upload()

df = spark.read.csv(
    "/content/global_cars_enhanced.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df.show(5)

Saving global_cars_enhanced.csv to global_cars_enhanced.csv
+--------+--------+----------------+---------+---------+------------+---------+----------+----------------+---------+---------------------+-------+--------------+---------+------------+----------------+
|  Car_ID|   Brand|Manufacture_Year|Body_Type|Fuel_Type|Transmission|Engine_CC|Horsepower|Mileage_km_per_l|Price_USD|Manufacturing_Country|Car_Age|Price_Category|HP_per_CC|Age_Category|Efficiency_Score|
+--------+--------+----------------+---------+---------+------------+---------+----------+----------------+---------+---------------------+-------+--------------+---------+------------+----------------+
|CAR_0001|Mercedes|            2006|      SUV|   Petrol|      Manual|     4089|       547|              17|    73407|                  USA|     20|       Premium|   0.1338|         Old|            0.35|
|CAR_0002|  Nissan|            2023|    Coupe|   Petrol|   Automatic|     4618|       167|              25|    79370|           

Kode ini digunakan untuk membuat kolom baru bernama Price_Category berdasarkan rentang harga mobil pada kolom Price_USD.

Proses ini merupakan tahap data transformation untuk mengubah data numerik menjadi kategori klasifikasi.

In [ ]:
df = df.withColumn(
    "Price_Category",
    when(df.Price_USD < 30000, "Budget")
    .when((df.Price_USD >= 30000) & (df.Price_USD < 60000), "Mid-Range")
    .when((df.Price_USD >= 60000) & (df.Price_USD < 90000), "Premium")
    .otherwise("Luxury")
)

categorical_cols Kode ini digunakan untuk mendefinisikan fitur kategorikal, yaitu data berbentuk kategori atau teks yang memerlukan proses encoding sebelum digunakan dalam machine learning.

numeric_cols Kode ini digunakan untuk mendefinisikan fitur numerik, yaitu data berbentuk angka yang akan digunakan sebagai input model klasifikasi.

In [ ]:
categorical_cols = [
    "Brand",
    "Body_Type",
    "Fuel_Type",
    "Transmission",
    "Manufacturing_Country",
    "Age_Category"
]

numeric_cols = [
    "Engine_CC",
    "Horsepower",
    "Mileage_km_per_l",
    "Car_Age",
    "HP_per_CC",
    "Efficiency_Score"
]

Kode ini digunakan untuk mengubah data kategorikal pada kolom Price_Category menjadi format numerik pada kolom label.

Proses ini diperlukan karena algoritma machine learning hanya dapat memproses data dalam bentuk numerik.

In [ ]:
label_indexer = StringIndexer(
    inputCol="Price_Category",
    outputCol="label"
)

Kode ini digunakan untuk mengubah seluruh fitur kategorikal menjadi format numerik menggunakan StringIndexer.

Hasil konversi disimpan pada kolom baru dengan tambahan nama _Index.

Kode ini digunakan untuk melakukan proses One Hot Encoding pada data kategorikal agar setiap kategori direpresentasikan dalam bentuk vektor biner. Proses ini membantu model machine learning menghindari bias terhadap urutan kategori.

In [ ]:
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_Index"
    ) for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=col + "_Index",
        outputCol=col + "_Vec"
    ) for col in categorical_cols
]

assembler_inputs digunakan untuk menggabungkan seluruh fitur kategorikal yang telah diencoding dengan fitur numerik ke dalam satu daftar input.

Kode ini digunakan untuk menggabungkan seluruh fitur menjadi satu vektor pada kolom features, yang nantinya digunakan sebagai input utama model machine learning.

In [ ]:
assembler_inputs = \
    [col + "_Vec" for col in categorical_cols] + numeric_cols

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

Kode ini digunakan untuk melakukan normalisasi fitur agar seluruh data memiliki skala yang lebih seimbang sebelum diproses oleh model machine learning.

In [ ]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

Kode ini digunakan untuk membagi dataset menjadi data training sebesar 80% dan data testing sebesar 20%.

In [ ]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

Kode ini digunakan untuk membangun model klasifikasi Random Forest dengan menggunakan fitur yang telah dinormalisasi pada kolom scaled_features.

In [ ]:
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100
)

Kode ini digunakan untuk membangun pipeline machine learning yang menggabungkan seluruh tahapan preprocessing dan model Random Forest dalam satu alur kerja terintegrasi.

Pipeline mencakup proses indexing, encoding, pembentukan fitur, normalisasi data, hingga pelatihan model klasifikasi.

In [ ]:
pipeline_rf = Pipeline(stages=
    indexers +
    encoders +
    [
        label_indexer,
        assembler,
        scaler,
        rf
    ]
)

Kode ini digunakan untuk melatih pipeline Random Forest menggunakan data training sehingga model dapat mempelajari pola dari dataset.

In [ ]:
model_rf = pipeline_rf.fit(train_data)

Kode ini digunakan untuk melakukan prediksi terhadap data testing menggunakan model yang telah dilatih sebelumnya.

In [ ]:
pred_rf = model_rf.transform(test_data)

Kode ini digunakan untuk menghitung dan menampilkan total waktu yang dibutuhkan selama proses training model berlangsung.

In [ ]:
# ======================================
# TRAINING
# ======================================
start_train = time.perf_counter()

model_rf = pipeline_rf.fit(train_data)

end_train = time.perf_counter()

print(f"Waktu Training: {end_train - start_train:.2f} detik")

Waktu Training: 7.57 detik


Kode tersebut digunakan untuk mengukur performa waktu prediksi model Random Forest menggunakan Apache Spark. Proses dimulai dengan pencatatan waktu awal, dilanjutkan prediksi data testing, eksekusi proses Spark, hingga pencatatan waktu akhir untuk menghitung total durasi prediksi model.

In [ ]:

# ======================================
# PREDIKSI
# ======================================
start_pred = time.perf_counter()

pred_rf = model_rf.transform(test_data)

# action Spark
pred_rf.count()

end_pred = time.perf_counter()

print(f"Waktu Prediksi: {end_pred - start_pred:.2f} detik")

Waktu Prediksi: 1.02 detik


Kode tersebut digunakan untuk mengevaluasi performa model klasifikasi Random Forest menggunakan metrik accuracy. Evaluasi dilakukan dengan membandingkan hasil prediksi model terhadap label asli pada data testing, kemudian menghasilkan nilai akurasi sebesar 23.40%.


In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator_acc.evaluate(pred_rf)

print("Accuracy:", accuracy)

Accuracy: 0.23404255319148937


Kode tersebut digunakan untuk melakukan evaluasi model Random Forest dengan menghitung nilai accuracy serta mengukur waktu evaluasi model. Proses ini bertujuan untuk mengetahui tingkat performa dan efisiensi model dalam melakukan klasifikasi data.


In [ ]:
# ======================================
# EVALUASI
# ======================================
start_eval = time.perf_counter()

accuracy = evaluator_acc.evaluate(pred_rf)

end_eval = time.perf_counter()

print("Accuracy:", accuracy)
print(f"Waktu Evaluasi: {end_eval - start_eval:.2f} detik")

Accuracy: 0.23404255319148937
Waktu Evaluasi: 0.44 detik
